# Employee Data Cleaning Pipeline

# 01 Duplicate Detection and Consolidation

The dataset contains 96,900 employee records from three HR sources.

The goal is to identify duplicate employees, recover missing information, and remove redundant records without losing valid data.

## 1. Preparation

In [1]:
import pandas as pd

df = pd.read_csv("employee_pay_equity_messy.csv")

print(f"Dataset shape: {df.shape}")
print(f"Unique employee IDs: {df['employee_id'].nunique():,}")

Dataset shape: (96900, 19)
Unique employee IDs: 96,900


## 2. Duplicate Detection

All employee IDs are unique, but some contain a `_DUP` suffix. These records may represent duplicated employees.

In [2]:
dup_mask = df["employee_id"].str.endswith("_DUP", na=False)

dup_records = df.loc[dup_mask].copy()

print(f"Potential duplicate records: {len(dup_records):,}")
print(f"Percentage: {dup_mask.mean():.2%}")

display(dup_records.head(10))

Potential duplicate records: 1,900
Percentage: 1.96%


,employee_id,full_name,source_dataset,age,gender,ethnicity,education_level,years_experience,tenure_years,department,job_title,country,remote_status,performance_rating,performance_review_score,hire_date,currency,salary_raw,pay_band
91,LEG-38343_DUP,Alex Montgomery,acquired_company_legacy,24.0,male,White,Associate Degree,2.0,2.0,it,IT Support Specialist,UK,remote,Meets Expectations,4.6,05-27-2024,GBP,28446,Band 1
94,E112941_DUP,Jennifer Henderson,hris_export_2023,30.0,M,NaN,MSc,12.0,0.0,Marketing,Marketing Manager,Brazil,REMOTE,NaN,4.6,2026/01/20,BRL,"222,281.50",Band 2
95,E169093_DUP,David Scott,hris_export_2023,39.0,MALE,Hispanic or Latino,MSc,20.0,4.0,Legal,General Counsel,France,REMOTE,Meets Expectations,NaN,2022/07/08,EUR,"31,208.18",Band 1
119,E140444_DUP,Mark Mcguire,survey_response_2024,28.0,F,Black or African American,PhD.,10.0,7.0,Engineering,DevOps Engineer,United Kingdom,On site,meets,3.0,15/05/2019,GBP,"£26,660",Band 1
183,E143987_DUP,Manuel Garcia,hris_export_2023,27.0,FEMALE,Hispanic or Latino,NaN,9.0,6.0,Customer Support,Support Team Lead,Nigeria,Remote,meets,2.8,06 Jan 2020,NGN,"63,536,395.90",Band 2
289,E121287_DUP,Christine Frazier,survey_response_2024,34.0,Male,NaN,BSc,16.0,2.0,IT,IT Manager,Kenya,Hybrid,Meets Expectations,4.1,2024/06/17,KES,"7,776,870 KES",Band 3
363,E137237_DUP,Ashley Brown,survey_response_2024,42.0,FEMALE,White,NaN,21.0,0.0,HR,HR Coordinator,Germany,On site,NaN,5.0,2026/02/20,EUR,"€21,558",Band 1
394,E116947_DUP,Evelyn Wilkerson,hris_export_2023,34.0,Man,Asian,MSc,16.0,3.0,Marketing,Marketing Manager,Netherlands,Remote,Meets Expectations,NaN,2023-06-18,EUR,"41,993.12",Band 2
410,E179401_DUP,Tonya Sloan,survey_response_2024,37.0,Male,White,PhD,19.0,6.0,Data Science,Head of Data,South Africa,remote,Below Expectations,2.2,08/04/2020,ZAR,"1,222,648 ZAR",Band 3
434,E150610_DUP,Kendra Lindsey,hris_export_2023,26.0,FEMALE,NaN,Associate Degree,8.0,7.0,HR,HR Business Partner,South Africa,Onsite,Exceeds Expectations,4.4,11-20-2018,ZAR,"562,073.48",Band 1


## 3. Duplicate Matching

Remove the `_DUP` suffix and match each record with its original employee ID.

In [3]:
dup_records["original_id"] = dup_records["employee_id"].str.removesuffix("_DUP")

original_records = df.set_index("employee_id")

matched_ids = dup_records["original_id"].isin(original_records.index)

print(f"Matched original records: {matched_ids.sum():,}")
print(f"Unmatched records: {(~matched_ids).sum():,}")

Matched original records: 1,900
Unmatched records: 0


In [4]:
sample_ids = dup_records.loc[matched_ids, "original_id"].head(5)

comparison = df[
    df["employee_id"].isin(sample_ids)
    | df["employee_id"].isin(sample_ids + "_DUP")
]

display(comparison.sort_values("employee_id").T)

,84560,94,10308,119,33884,183,89126,95,68307,91
employee_id,E112941,E112941_DUP,E140444,E140444_DUP,E143987,E143987_DUP,E169093,E169093_DUP,LEG-38343,LEG-38343_DUP
full_name,Jennifer Henderson,Jennifer Henderson,Mark Mcguire,Mark Mcguire,Manuel Garcia,Manuel Garcia,David Scott,David Scott,Alex Montgomery,Alex Montgomery
source_dataset,hris_export_2023,hris_export_2023,survey_response_2024,survey_response_2024,hris_export_2023,hris_export_2023,hris_export_2023,hris_export_2023,acquired_company_legacy,acquired_company_legacy
age,30.0,30.0,28.0,28.0,27.0,27.0,39.0,39.0,24.0,24.0
gender,M,M,F,F,FEMALE,FEMALE,MALE,MALE,male,male
ethnicity,NaN,NaN,Black or African American,Black or African American,Hispanic or Latino,Hispanic or Latino,Hispanic or Latino,Hispanic or Latino,White,White
education_level,MSc,MSc,PhD.,PhD.,NaN,NaN,MSc,MSc,Associate Degree,Associate Degree
years_experience,12.0,12.0,10.0,10.0,9.0,9.0,20.0,20.0,2.0,2.0
tenure_years,0.0,0.0,7.0,7.0,6.0,6.0,4.0,4.0,2.0,2.0
department,Marketing,Marketing,Engineering,Engineering,Customer Support,Customer Support,Legal,Legal,it,it


## 4. Duplicate Comparison

Compare matched records across all columns, excluding employee IDs.

Two missing values in the same field are treated as equal.

In [5]:
columns = df.columns.drop("employee_id")

matched_duplicates = dup_records.loc[matched_ids].set_index("original_id")

originals = original_records.loc[matched_duplicates.index]

identical = (
    matched_duplicates[columns]
    .eq(originals[columns])
    .fillna(False)
    | (
        matched_duplicates[columns].isna()
        & originals[columns].isna()
    )
).all(axis=1)

print(f"Identical duplicate records: {identical.sum():,}")
print(f"Records with differences: {(~identical).sum():,}")

Identical duplicate records: 1,653
Records with differences: 247


### 4.1 Investigate Differences

1,653 records are identical to their originals, while 247 contain differences.

I'll examine these differences to determine whether they represent missing information, formatting issues, or conflicting values.

In [6]:
conflicting_ids = identical.index[~identical]

conflicting_duplicates = matched_duplicates.loc[conflicting_ids, columns]
conflicting_originals = originals.loc[conflicting_ids, columns]

differences = (
    conflicting_duplicates.ne(conflicting_originals)
    & ~(
        conflicting_duplicates.isna()
        & conflicting_originals.isna()
    )
)

difference_summary = differences.sum().sort_values(ascending=False)

display(difference_summary[difference_summary > 0])

job_title                   101
performance_review_score     45
age                          39
tenure_years                 38
years_experience             32
dtype: int64

In [7]:
sample_ids = conflicting_ids[:50]

examples = []

for employee_id in sample_ids:
    original = conflicting_originals.loc[employee_id]
    duplicate = conflicting_duplicates.loc[employee_id]

    changed_columns = differences.loc[employee_id]
    changed_columns = changed_columns[changed_columns].index

    for column in changed_columns:
        original_value = original[column]
        duplicate_value = duplicate[column]

        examples.append({
            "employee_id": employee_id,
            "column": column,
            "original_value": original_value.lower() if isinstance(original_value, str) else original_value,
            "duplicate_value": duplicate_value.lower() if isinstance(duplicate_value, str) else duplicate_value
        })

comparison_df = pd.DataFrame(examples)

display(comparison_df)

,employee_id,column,original_value,duplicate_value
0,E137237,job_title,hr coordinator,hr coordinator
1,E179401,performance_review_score,NaN,2.2
2,E159320,years_experience,NaN,25.0
3,E116519,performance_review_score,NaN,3.2
4,E100980,age,NaN,39.0
5,E178329,years_experience,NaN,10.0
6,LEG-92619,tenure_years,1.0,NaN
7,LEG-83480,job_title,controller,controller
8,E133027,job_title,support agent,support agent
9,E121727,job_title,cmo,cmo


### 4.2 Classify Differences

Classify each difference into three groups:

- **Missing values:** One record contains information missing from the other.
- **Formatting differences:** Values differ only in capitalization or whitespace.
- **Conflicting values:** Both records contain different non-null values.

In [8]:
summary = {
    "complementary_missing": 0,
    "formatting_differences": 0,
    "conflicting_values": 0
}

for employee_id in conflicting_ids:
    original = conflicting_originals.loc[employee_id]
    duplicate = conflicting_duplicates.loc[employee_id]

    for column in columns:
        a = original[column]
        b = duplicate[column]

        if pd.isna(a) and pd.isna(b):
            continue

        if pd.isna(a) != pd.isna(b):
            summary["complementary_missing"] += 1

        elif isinstance(a, str) and isinstance(b, str):
            if a == b:
                continue
            elif a.strip().casefold() == b.strip().casefold():
                summary["formatting_differences"] += 1
            else:
                summary["conflicting_values"] += 1

        elif a != b:
            summary["conflicting_values"] += 1

display(pd.Series(summary, name="Count").to_frame())

,Count
complementary_missing,154
formatting_differences,101
conflicting_values,0


## 5. Consolidation

Merge duplicate records using the following rules:

1. Keep the original employee ID.
2. Recover missing values from the matching duplicate.
3. Preserve existing values when differences are limited to formatting.
4. Remove redundant records after consolidation.

In [9]:
df_clean = df.copy()

original_ids = dup_records["original_id"]

assert original_ids.isin(df_clean["employee_id"]).all()
assert original_ids.is_unique

df_clean = df_clean.set_index("employee_id")

duplicates = df_clean.loc[dup_records["employee_id"]].copy()
duplicates.index = original_ids.to_numpy()

originals = df_clean.loc[original_ids].copy()
originals.index = original_ids.to_numpy()

merged = originals.combine_first(duplicates)

df_clean.loc[original_ids, merged.columns] = merged.to_numpy()

df_clean = df_clean.drop(index=dup_records["employee_id"])

df_clean = df_clean.reset_index()

print(f"Original rows: {len(df):,}")
print(f"Cleaned rows: {len(df_clean):,}")
print(f"Removed duplicates: {len(df) - len(df_clean):,}")

Original rows: 96,900
Cleaned rows: 95,000
Removed duplicates: 1,900


## 6. Validation

Verify record counts, employee ID uniqueness, duplicate removal, and recovered missing values.

In [10]:
assert len(df_clean) == len(df) - 1900
assert df_clean["employee_id"].is_unique
assert not df_clean["employee_id"].str.endswith("_DUP").any()

missing_before = df.isna().sum().sum()
missing_after = df_clean.isna().sum().sum()

print(f"Rows removed: {len(df) - len(df_clean):,}")
print(f"Missing values before: {missing_before:,}")
print(f"Missing values after: {missing_after:,}")

recovered_count = (
    originals.isna() & duplicates.notna()).sum().sum()

print(f"Missing values recovered: {recovered_count:,}")
print("Validation passed.")

Rows removed: 1,900
Missing values before: 67,549
Missing values after: 66,131
Missing values recovered: 76
Validation passed.


## 7. Export

In [11]:
from pathlib import Path

output_path = Path("data/processed/01_duplicates_resolved.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(output_path, index=False)

print(f"Saved: {output_path}")

Saved: data/processed/01_duplicates_resolved.csv


## 8. Summary

- **96,900** original records.
- **1,900** duplicate records identified and removed.
- **154** missing values recovered.
- **101** formatting differences identified.
- **0** conflicting non-null values.
- **95,000** unique employee records in the final dataset.

Original employee IDs and valid information were preserved.